# Local HF Endpoint Book Summarization

Run this notebook locally. The tokenizer, GCS loading, chapter splitting, and result saving run on this machine. The summarization model inference runs on a Hugging Face Inference Endpoint.

## Environment

Before running, set these environment variables in your shell or a local `.env` file:

```bash
export HF_TOKEN="..."
export GOOGLE_APPLICATION_CREDENTIALS="/absolute/path/to/credentials.json"
# Or set GOOGLE_APPLICATION_CREDENTIALS in src/AudioBooks/.env.
export GCS_BUCKET="gutenberg-books"
```

In [ ]:
%pip install -U transformers sentencepiece huggingface_hub google-cloud-storage python-dotenv psutil torch

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import sys
import time

from pathlib import Path

from dotenv import dotenv_values, load_dotenv
from huggingface_hub import InferenceClient, create_inference_endpoint, get_inference_endpoint
from transformers import AutoTokenizer

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((root for root in search_roots if (root / "src" / "AudioBooks").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Launch this notebook from inside the AudioBooks repository.")
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
DOTENV_PATH = REPO_ROOT / "src" / "AudioBooks" / ".env"
if not DOTENV_PATH.is_file():
    DOTENV_PATH = None

dotenv_values_map = dotenv_values(DOTENV_PATH) if DOTENV_PATH else {}
load_dotenv(DOTENV_PATH, override=True) if DOTENV_PATH else load_dotenv(override=True)

google_credentials = dotenv_values_map.get("GOOGLE_APPLICATION_CREDENTIALS") or os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
if google_credentials:
    credentials_path = Path(google_credentials).expanduser()
    if DOTENV_PATH and not credentials_path.is_absolute():
        credentials_path = DOTENV_PATH.parent / credentials_path
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path)


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
GCS_BUCKET = os.environ.get("GCS_BUCKET", "gutenberg-books")
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_API_TOKEN")
GOOGLE_APPLICATION_CREDENTIALS = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN (or HF_API_TOKEN) in your environment or .env file.")

if not GOOGLE_APPLICATION_CREDENTIALS:
    raise RuntimeError("Set GOOGLE_APPLICATION_CREDENTIALS to your GCS service-account JSON path.")

credentials_path = Path(GOOGLE_APPLICATION_CREDENTIALS).expanduser()
if not credentials_path.exists():
    raise FileNotFoundError(f"Google credentials file does not exist: {credentials_path}")

print(f".env: {DOTENV_PATH}")
print(f"Bucket: {GCS_BUCKET}")
print(f"Model: {MODEL_ID}")
print(f"GOOGLE_APPLICATION_CREDENTIALS: {credentials_path}")

In [ ]:
import AudioBooks.BookSummary.summarizer as ai_summarizer

ai_summarizer = importlib.reload(ai_summarizer)

_make_gcs_client = ai_summarizer._make_gcs_client
_load_book_record = ai_summarizer._load_book_record
summarize_book = ai_summarizer.summarize_book
_semantic_similarity = ai_summarizer._semantic_similarity
_lexical_similarity = ai_summarizer._lexical_similarity
_max_nli_contradiction = ai_summarizer._max_nli_contradiction

gcs_client = _make_gcs_client(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"))

id_map_blob = gcs_client.bucket(GCS_BUCKET).blob("book-desc/gutenberg-id-map.json")
id_map: dict[str, int] = json.loads(id_map_blob.download_as_text(encoding="utf-8"))
all_book_ids = sorted(set(id_map.values()))

print(f"Found {len(all_book_ids)} books in GCS bucket {GCS_BUCKET!r}")

In [ ]:
# Hugging Face Inference Endpoint configuration.
# For a quick smoke test, set ENDPOINT_REPOSITORY = "gpt2" and use CPU.
# For Qwen 7B, use a GPU instance and confirm endpoint billing is enabled.
ENDPOINT_NAME = "audiobook-summary-qwen25-7b"
ENDPOINT_REPOSITORY = MODEL_ID
ENDPOINT_FRAMEWORK = "pytorch"
ENDPOINT_TASK = "text-generation"
ENDPOINT_ACCELERATOR = "gpu"
ENDPOINT_VENDOR = "aws"
ENDPOINT_REGION = "us-east-1"
ENDPOINT_INSTANCE_SIZE = "x1"
ENDPOINT_INSTANCE_TYPE = "nvidia-a100"
ENDPOINT_TYPE = "authenticated"
CREATE_ENDPOINT_IF_MISSING = True


def load_or_create_endpoint():
    try:
        endpoint = get_inference_endpoint(ENDPOINT_NAME, token=HF_TOKEN)
        print(f"Using existing endpoint: {ENDPOINT_NAME}")
    except Exception as exc:
        if not CREATE_ENDPOINT_IF_MISSING:
            raise
        print(f"Creating endpoint {ENDPOINT_NAME}: {exc}")
        endpoint = create_inference_endpoint(
            ENDPOINT_NAME,
            repository=ENDPOINT_REPOSITORY,
            framework=ENDPOINT_FRAMEWORK,
            task=ENDPOINT_TASK,
            accelerator=ENDPOINT_ACCELERATOR,
            vendor=ENDPOINT_VENDOR,
            region=ENDPOINT_REGION,
            instance_size=ENDPOINT_INSTANCE_SIZE,
            instance_type=ENDPOINT_INSTANCE_TYPE,
            type=ENDPOINT_TYPE,
            token=HF_TOKEN,
        )

    endpoint.wait()
    print(f"Endpoint URL: {endpoint.url}")
    return endpoint


endpoint = load_or_create_endpoint()
hf_client = InferenceClient(model=endpoint.url, token=HF_TOKEN)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# Tokenizer runs locally only for prompt/chunk sizing. Model generation runs on the HF endpoint.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token or "[PAD]"

# Chunks shorter than this token count are skipped (usually just headings with no body).
MIN_CHUNK_TOKENS = 20


def _truncate_at_prompt_marker(text: str) -> str:
    """Cut off any hallucinated continuation at the first ### marker."""
    idx = text.find("###")
    return text[:idx].strip() if idx != -1 else text.strip()


def _call_hf_endpoint(prompt: str, *, max_new_tokens: int) -> str:
    print(
        f"Calling HF endpoint {ENDPOINT_NAME} for {len(prompt):,} prompt chars, "
        f"max_new_tokens={max_new_tokens}",
        flush=True,
    )
    for attempt in range(3):
        try:
            result = hf_client.text_generation(
                prompt,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.05,
                return_full_text=False,
            )
            result = _truncate_at_prompt_marker(result)
            print(f"RAW MODEL OUTPUT: {result!r}", flush=True)
            return result
        except Exception as exc:
            if attempt == 2:
                raise
            wait_seconds = 5 * (attempt + 1)
            print(f"Endpoint generation failed ({exc}); retrying in {wait_seconds}s", flush=True)
            time.sleep(wait_seconds)


def _remote_generate_text(model, tokenizer, prompt: str, *, device, max_input_tokens: int, max_new_tokens: int) -> str:
    input_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=max_input_tokens,
    ).input_ids
    truncated_prompt = tokenizer.decode(input_ids, skip_special_tokens=True)
    generated = _call_hf_endpoint(truncated_prompt, max_new_tokens=max_new_tokens)
    return ai_summarizer._strip_prompt(generated)


def _remote_generate_batch(model, tokenizer, prompts: list[str], *, device, max_input_tokens: int, max_new_tokens: int) -> list[str]:
    # Filter out trivial prompts (heading-only chunks) before sending to the endpoint.
    results = [""] * len(prompts)
    work = [
        (i, p) for i, p in enumerate(prompts)
        if len(tokenizer(p, add_special_tokens=False).input_ids) >= MIN_CHUNK_TOKENS
    ]
    skipped = len(prompts) - len(work)
    if skipped:
        print(f"  skipping {skipped} trivial chunk(s) with < {MIN_CHUNK_TOKENS} tokens", flush=True)
    if not work:
        return results
    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        futures = {
            executor.submit(
                _remote_generate_text,
                model, tokenizer, prompt,
                device=device,
                max_input_tokens=max_input_tokens,
                max_new_tokens=max_new_tokens,
            ): idx
            for idx, prompt in work
        }
        for future, idx in futures.items():
            results[idx] = future.result()
    return results


ai_summarizer._generate_text = _remote_generate_text
ai_summarizer._generate_batch = _remote_generate_batch

model = hf_client
print("HF cloud generation is configured. No local generation model was loaded.")

In [ ]:
from pathlib import Path
import AudioBooks.BookSummary.summarizer as _summarizer_mod

# Summarization configuration.
# Set BOOK_ID_TO_SUMMARIZE to one internal book_id for a single-book run.
# Leave it as None to resume through the first MAX_BOOKS unprocessed books.
BOOK_ID_TO_SUMMARIZE = 4037
MAX_BOOKS = 20
RUN_VALIDATION = True
SEMANTIC_THRESHOLD = 0.60
LEXICAL_FLOOR = 0.35
NLI_THRESHOLD = 0.50
CHUNK_TOKENS = 4096
CHUNK_OVERLAP = 400
REDUCE_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = 512
REDUCE_MAX_NEW_TOKENS = 768
PROFILE_MAX_NEW_TOKENS = 1024   # tokens for character/figure profile generation
MAX_CHAPTERS = None
MAX_CHUNKS_PER_CHAPTER = None
BATCH_SIZE = 4

# Results live next to the BookSummary package (src/AudioBooks/BookSummary/Artifacts)
# regardless of the notebook's working directory, matching spot_vm_summarize.py
# and runpod_summarize.py.
ARTIFACTS_DIR = Path(_summarizer_mod.__file__).resolve().parent / "Artifacts"
OUTPUT_PATH = ARTIFACTS_DIR / "summary_results_local.jsonl"
CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
embedding_tokenizer = embedding_model = None
nli_tokenizer = nli_model = None
if RUN_VALIDATION:
    embedding_tokenizer, embedding_model = ai_summarizer._load_embedding_model(
        ai_summarizer.DEFAULT_EMBEDDING_MODEL_ID,
        HF_TOKEN,
    )
    nli_tokenizer, nli_model = ai_summarizer._load_nli_model(
        ai_summarizer.DEFAULT_NLI_MODEL_ID,
        HF_TOKEN,
    )


def summarize_one_book_via_hf_endpoint(book_id: int) -> dict | None:
    print(f"\nbook_id={book_id}", flush=True)
    try:
        book, book_desc_summary = _load_book_record(gcs_client, GCS_BUCKET, book_id)
    except FileNotFoundError as exc:
        print(f"  SKIP: {exc}", flush=True)
        return None

    mode = "verify" if book_desc_summary and RUN_VALIDATION else "generate"
    print(f"  mode={mode} title={book.title!r} category={book.category or '(unknown)'!r}", flush=True)
    print(f"  subjects={book.subjects}", flush=True)
    print(f"  generation_endpoint={endpoint.url}", flush=True)

    checkpoint_path = str(CHECKPOINT_DIR / f"checkpoint_{book_id}.jsonl")

    result = summarize_book(
        model,
        tokenizer,
        book,
        device=None,
        chunk_tokens=CHUNK_TOKENS,
        chunk_overlap=CHUNK_OVERLAP,
        reduce_input_tokens=REDUCE_INPUT_TOKENS,
        max_new_tokens=MAX_NEW_TOKENS,
        reduce_max_new_tokens=REDUCE_MAX_NEW_TOKENS,
        batch_size=BATCH_SIZE,
        max_chapters=MAX_CHAPTERS,
        max_chunks_per_chapter=MAX_CHUNKS_PER_CHAPTER,
        story_so_far_tokens=REDUCE_MAX_NEW_TOKENS,
        checkpoint_path=checkpoint_path,
        profile_max_new_tokens=PROFILE_MAX_NEW_TOKENS,
    )

    result["mode"] = mode
    result["book_desc_summary"] = book_desc_summary
    result["semantic_score"] = None
    result["lexical_score"] = None
    result["nli_contradiction_score"] = None
    result["similarity_pass"] = None

    # Print the final story-so-far (last chapter's running summary).
    if result.get("chapters"):
        final_story_so_far = result["chapters"][-1].get("story_so_far", "")
        if final_story_so_far:
            print("\nSTORY SO FAR (end of book)")
            print(final_story_so_far)

    print("\nFINAL SUMMARY")
    print(result["final_summary"])

    profiles = result.get("character_profiles", "")
    if profiles:
        profile_label = "NARRATOR PROFILE" if result.get("category") == "practical" else "CHARACTER PROFILES"
        print(f"\n{profile_label}  [category={result.get('category', '?')!r}]")
        print(profiles)

    if RUN_VALIDATION and book_desc_summary:
        semantic = _semantic_similarity(embedding_tokenizer, embedding_model, result["final_summary"], book_desc_summary)
        lexical = _lexical_similarity(result["final_summary"], book_desc_summary)
        contradiction = _max_nli_contradiction(nli_tokenizer, nli_model, result["final_summary"], book_desc_summary)
        sem_pass = semantic >= SEMANTIC_THRESHOLD
        lex_pass = lexical >= LEXICAL_FLOOR
        nli_pass = contradiction < NLI_THRESHOLD
        sim_pass = sem_pass and (lex_pass or nli_pass)
        result.update({
            "semantic_score": semantic,
            "lexical_score": lexical,
            "nli_contradiction_score": contradiction,
            "similarity_pass": sim_pass,
        })
        print(
            f"  semantic={semantic:.3f} lexical={lexical:.3f}"
            f" nli_contradiction={contradiction:.3f} pass={sim_pass}",
            flush=True,
        )

    return result

In [ ]:
processed_ids: set[int] = set()
if OUTPUT_PATH.exists() and BOOK_ID_TO_SUMMARIZE is None:
    for line in OUTPUT_PATH.read_text(encoding="utf-8").splitlines():
        try:
            processed_ids.add(json.loads(line)["book_id"])
        except Exception:
            pass

if BOOK_ID_TO_SUMMARIZE is not None:
    book_ids_to_run = [BOOK_ID_TO_SUMMARIZE]
else:
    book_ids_to_run = [book_id for book_id in all_book_ids if book_id not in processed_ids]
    if MAX_BOOKS is not None:
        book_ids_to_run = book_ids_to_run[:MAX_BOOKS]

print(f"Already processed: {len(processed_ids)} books")
print(f"Running: {len(book_ids_to_run)} books via HF endpoint {ENDPOINT_NAME}")

for idx, book_id in enumerate(book_ids_to_run, 1):
    print(f"\n[{idx}/{len(book_ids_to_run)}]", flush=True)
    try:
        result = summarize_one_book_via_hf_endpoint(book_id)
    except Exception as exc:
        print(f"  ERROR summarizing: {exc}", flush=True)
        continue

    if result is None:
        continue

    with OUTPUT_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(result, ensure_ascii=False) + "\n")
    print(f"  saved -> {OUTPUT_PATH}", flush=True)

print("\nDone.")